In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau, wilcoxon, kruskal, mannwhitneyu
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from statsmodels.stats.multitest import multipletests
import re
import unicodedata
import math
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.contingency_tables import mcnemar
import scikit_posthocs as sp

In [2]:
gcms_df = pd.read_csv('seed_derived_gcms_data.csv', engine='python')
seed_traits_df = pd.read_csv('seed_traits.csv', engine='python')
seed_area_df = pd.read_csv('seed_area_measurements.csv', engine='python')

In [3]:
# Function to approximate 3D seed surface area from projected area using species-specific correction factors
def surface_area_approximation(
    df,
    species_col="species",
    projected_area_col="projected_surface_area/mm^2",
    new_col="approx_surface_area/mm^2",
):
    """
    Multiply projected surface area by species-specific correction factors
    to approximate true 3D surface area.

    Parameters
    ----------
    df : DataFrame
        Must contain species_col and projected_area_col.
    custom_factors : dict or None
        Optional dictionary mapping species -> correction factor.
        If None, default factors based on morphology are used.

    Returns
    -------
    DataFrame with additional column new_col.
    """

    # Default morphology-based factors (transparent + editable)
    correction_factors = {
        "Lactuca serriola": 2.0,             # very flat
        "Daucus carota": 2.2,                # moderately flat
        "Vaccinium oxycoccus": 2.5,
        "Papaver rhoeas": 3.0,               # roughly kidney-shaped
        "Mesembryanthemum crystallinum": 3.0,
        "Ficus sycomorus": 3.0,
        "Trifolium pratense": 3.4,           # round/ellipsoid
    }

    df = df.copy()

    # Map species to factor
    df["correction_factor"] = df[species_col].map(correction_factors)

    # Warn if any species missing
    if df["correction_factor"].isna().any():
        missing = df.loc[df["correction_factor"].isna(), species_col].unique()
        raise ValueError(f"No correction factor provided for: {missing}")

    # Apply correction
    df[new_col] = df[projected_area_col] * df["correction_factor"]

    return df, correction_factors

seed_area_df, corr_factors = surface_area_approximation(seed_area_df)

In [4]:
def build_replicate_aligned_totals(
    df,
    traits_df,
    *,
    treatment,
    trait_col,
    subset_b=None,                # callable or {"ids":...} or {"names":..., "match":...} or {"query":...}
    agg="mean",                   # used later; keep for symmetry
    species_col="species",
    treatment_col="treatment",
    replicate_col="replicate",
    compound_id_col="compound_id",
    compound_name_col="compound_name",
    signal_col="area/sample_mass",
    drop_empty_species=True,
    empty_label="Empty",
):
    d = df.copy()
    if drop_empty_species:
        d = d[d[species_col] != empty_label]
    d = d[d[treatment_col] == treatment].copy()

    # Replicate totals for A (all compounds)
    rep_a = (
        d.groupby([species_col, replicate_col], dropna=False)[signal_col]
         .sum()
         .rename("rep_a")
         .reset_index()
    )

    # Subset mask for B (markers)
    if subset_b is None:
        rep_b = rep_a.rename(columns={"rep_a": "rep_b"})
    else:
        if callable(subset_b):
            mask = subset_b(d)
        elif isinstance(subset_b, dict) and "ids" in subset_b:
            mask = d[compound_id_col].isin(set(subset_b["ids"]))
        elif isinstance(subset_b, dict) and "names" in subset_b:
            names = pd.Series(subset_b["names"]).astype(str)
            match = subset_b.get("match", "exact")
            if match == "exact":
                mask = d[compound_name_col].astype(str).isin(set(names))
            elif match == "contains":
                pat = "|".join(map(lambda s: str(s), names.tolist()))
                mask = d[compound_name_col].fillna("").astype(str).str.contains(pat, case=False, regex=True)
            else:
                raise ValueError("subset_b['match'] must be 'exact' or 'contains'")
        elif isinstance(subset_b, dict) and "query" in subset_b:
            mask = d.eval(subset_b["query"]).astype(bool)
        else:
            raise ValueError("subset_b must be callable, or dict with ids/names/query")

        rep_b = (
            d[mask]
            .groupby([species_col, replicate_col], dropna=False)[signal_col]
            .sum()
            .rename("rep_b")
            .reset_index()
        )

    # Align replicate grid: missing subset => 0
    rep = rep_a.merge(rep_b, on=[species_col, replicate_col], how="left")
    rep["rep_b"] = rep["rep_b"].fillna(0.0)

    # Add trait per species
    trait = traits_df.set_index(species_col)[trait_col].rename("trait")
    rep = rep.merge(trait.reset_index(), on=species_col, how="inner")

    return rep

In [5]:
# Create a global ID map for compounds
HYPHENS_RE = re.compile(r"[\u2010\u2011\u2012\u2013\u2014\u2212]")  # ‐-‒–—−
SQUARE_BRACKETS_RE = re.compile(r"\[[^\]]*\]")

# Remove things like (E), (Z), (E,Z), (R), (S), (all-Z), cis/trans etc
STEREO_PARENS_RE = re.compile(
    r"\(\s*(?:[ersz]|[ez]\s*,\s*[ez]|cis|trans|dl|d|l|all-z|all\s*-\s*z|e,e,e|e,z|z,e|e,e|z,z)\s*\)",
    flags=re.IGNORECASE,
)

DOT_GREEK_REPLACEMENTS = {
    r"\.alpha\.": " alpha ",
    r"\.beta\.":  " beta ",
    r"\.gamma\.": " gamma ",
    r"\.delta\.": " delta ",
}

GREEK_CHAR_MAP = {"α":"alpha","β":"beta","γ":"gamma","δ":"delta","ε":"epsilon","μ":"mu","ω":"omega"}
LEADING_STEREO_PREFIX_RE = re.compile(r"^(?:cis|trans|dl|d|l)\s*-\s*", flags=re.IGNORECASE)

def normalise_compound_name(s) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)

    s = unicodedata.normalize("NFKC", s).lower()
    s = HYPHENS_RE.sub("-", s)

    # greek chars
    for g, w in GREEK_CHAR_MAP.items():
        s = s.replace(g, w)

    # ".beta." style
    for pat, repl in DOT_GREEK_REPLACEMENTS.items():
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)

    # remove bracket blocks like [1R-(...)]
    s = SQUARE_BRACKETS_RE.sub(" ", s)

    # Remove parenthetical stereochem blocks that contain alpha/beta (or greek letters)
    s = re.sub(r"\([^)]*(?:alpha|beta|gamma|delta|α|β|γ|δ)[^)]*\)\s*-?", " ", s, flags=re.IGNORECASE)

    # remove small stereo parentheses
    s = STEREO_PARENS_RE.sub(" ", s)

    # drop leading "trans-" / "cis-" etc (outside parentheses)
    s = LEADING_STEREO_PREFIX_RE.sub("", s)

    # remove stray leading dots
    s = re.sub(r"^\.+", "", s)

    # keep letters/digits/hyphens/spaces, turn other punctuation into spaces
    s = re.sub(r"[^a-z0-9\-\s]", " ", s)

    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # KEY STEP: collapse spaces around hyphens so "beta -myrcene" -> "beta-myrcene"
    s = re.sub(r"\s*-\s*", "-", s)

    # strip trailing hyphens
    s = re.sub(r"-+$", "", s)

    return s

In [6]:
oil_markers = ['Nonadecane',
'Octadecane',
'Heneicosane',
'Eicosane',
'Henicos-1-ene',
'Octadecane, 3-methyl-',
'Heptadecane, 3-methyl-',
'Octadecane, 2-methyl-',
'Nonanoic acid',
'β-Myrcene',
'p-Cymene',
'2,6-Octadien-1-ol, 3,7-dimethyl-, acetate (Z)',
'Pentadecane',
'1-Tetradecanol',
'Geraniol',
'6-Hydroxy-3,7-dimethyl-2,7-octadienyl acetate (E)',
'2,6-Octadien-1-ol, 3,7-dimethyl-, acetate (Z)',
'Acetic acid, 3-methyl-6-oxo-hex-2-enyl ester'
]

oil_markers = [normalise_compound_name(c) for c in oil_markers]

In [7]:
ageing_markers = ['Hexadecanal',
'Pentadecanal',
'Butanoic acid, 2-methyl-',
'Acetone',
'2-Octenal (E)',
'2-Decenal (E)',
'Furan, 2-pentyl-',
'1-Hexanol',
'Benzaldehyde',
]

ageing_markers = [normalise_compound_name(c) for c in ageing_markers]

In [8]:
# Function to create species×treatment summary with signal totals and seed traits
def make_species_treatment_summary(
    gcms_df,
    seed_area_df,
    seed_traits_df,
    subset=None,
    expected_n_replicates=3,
    species_col="species",
    treatment_col="treatment",
    replicate_col="replicate",
    signal_col="area",
    n_seeds_col="n_seeds",
    sample_mass_col="sample_mass_mg",
    compound_id_col="compound_id",
    compound_name_col="compound_name",
    compound_norm_col="compound_norm",
    empty_label="Empty",
    # seed_area_df columns
    seed_area_species_col="species",
    projected_area_col="projected_surface_area/mm^2",
    approx_area_col="approx_surface_area/mm^2",
    # seed_traits_df columns
    seed_traits_species_col="species",
    seed_mass_col="seed_mass_mg",
    oil_low_col="oil_content_low",
    oil_high_col="oil_content_high",
    oil_mean_col="oil_content_mean",
    germination_change_col="germination_change",
):
    """
    Returns species×treatment summary with:
      - mean/std/min/max total signal per seed and per mass
      - n_replicates
      - n_compounds_detected_ge_2_reps
      - n_compounds_detected_3_reps
      - mean/std projected and approx seed surface area
      - species-level seed trait columns from seed_traits_df

    If a species×treatment has fewer than expected_n_replicates observed in gcms_df,
    missing replicates are added with total_signal = 0.
    """

    def make_mask(d):
        if subset is None:
            return pd.Series(True, index=d.index)
        if callable(subset):
            return pd.Series(subset(d), index=d.index)

        if isinstance(subset, dict):
            if "query" in subset:
                return d.eval(subset["query"]).astype(bool)

            if "ids" in subset:
                return d[compound_id_col].isin(set(subset["ids"]))

            if "names" in subset:
                names = [str(x) for x in subset["names"]]
                match = subset.get("match", "exact")
                s = d[compound_name_col].fillna("").astype(str)
                if match == "exact":
                    return s.isin(set(names))
                elif match == "contains":
                    m = pd.Series(False, index=d.index)
                    for nm in names:
                        m |= s.str.contains(nm, case=False, na=False, regex=False)
                    return m
                else:
                    raise ValueError("subset['match'] must be 'exact' or 'contains'")

            if "norm" in subset:
                norms = [str(x) for x in subset["norm"]]
                match = subset.get("match", "exact")
                s = d[compound_norm_col].fillna("").astype(str)
                if match == "exact":
                    return s.isin(set(norms))
                elif match == "contains":
                    m = pd.Series(False, index=d.index)
                    for nm in norms:
                        m |= s.str.contains(nm, case=False, na=False, regex=False)
                    return m
                else:
                    raise ValueError("subset['match'] must be 'exact' or 'contains'")

        raise ValueError("Bad subset spec")

    # ---- base: remove Empty once and use everywhere ----
    base = gcms_df[gcms_df[species_col] != empty_label].copy()

    # subset-applied detections for signal totals and compound counts
    d = base[make_mask(base)].copy()

    # ------------------------------------------------------------------
    # Compound counts by replicate support
    # ------------------------------------------------------------------
    # Count unique replicates in which each compound appears for each species×treatment
    compound_rep_support = (
        d.groupby([species_col, treatment_col, compound_id_col], dropna=False)[replicate_col]
         .nunique()
         .rename("n_reps_detected")
         .reset_index()
    )

    compound_counts = (
        compound_rep_support
        .groupby([species_col, treatment_col], dropna=False)
        .agg(
            n_compounds_detected_ge_2_reps=("n_reps_detected", lambda x: (x >= 2).sum()),
            n_compounds_detected_3_reps=("n_reps_detected", lambda x: (x== 3).sum()),
        )
        .reset_index()
    )

    # replicate totals for signal (subset)
    rep_sig = (
        d.groupby([species_col, treatment_col, replicate_col], dropna=False)[signal_col]
         .sum()
         .rename("total_signal")
         .reset_index()
    )

    # replicate grid for all observed replicates
    grid = (
        base.groupby([species_col, treatment_col, replicate_col], dropna=False)
            .agg(
                n_seeds=(n_seeds_col, "first"),
                sample_mass=(sample_mass_col, "first"),
            )
            .reset_index()
    )

    rep = grid.merge(rep_sig, on=[species_col, treatment_col, replicate_col], how="left")
    rep["total_signal"] = rep["total_signal"].fillna(0.0)

    # force numeric
    rep["n_seeds"] = pd.to_numeric(rep["n_seeds"], errors="coerce")
    rep["sample_mass"] = pd.to_numeric(rep["sample_mass"], errors="coerce")

    # per-replicate ratios
    rep["signal_per_seed"] = rep["total_signal"] / rep["n_seeds"]
    rep["signal_per_mass"] = rep["total_signal"] / rep["sample_mass"]

    # ------------------------------------------------------------------
    # Pad each species×treatment to expected_n_replicates with zero rows
    # ------------------------------------------------------------------
    padded_rows = []

    for (sp, tr), grp in rep.groupby([species_col, treatment_col], dropna=False):
        n_obs = len(grp)
        n_missing = max(0, expected_n_replicates - n_obs)

        if n_missing > 0:
            existing_labels = set(grp[replicate_col].astype(str))

            for j in range(n_missing):
                new_label = f"__added_zero_rep_{j+1}"
                while new_label in existing_labels:
                    new_label = f"__added_zero_rep_{j+1}_x"

                padded_rows.append({
                    species_col: sp,
                    treatment_col: tr,
                    replicate_col: new_label,
                    "n_seeds": np.nan,
                    "sample_mass": np.nan,
                    "total_signal": 0.0,
                    "signal_per_seed": 0.0,
                    "signal_per_mass": 0.0,
                })

    if padded_rows:
        rep = pd.concat([rep, pd.DataFrame(padded_rows)], ignore_index=True)

    # species×treatment mean/std/min/max across replicates
    out = (
        rep.groupby([species_col, treatment_col], dropna=False)
           .agg(
               mean_total_area_sample_mass=("signal_per_mass", "mean"),
               std_total_area_sample_mass=("signal_per_mass", "std"),
               min_total_area_sample_mass=("signal_per_mass", "min"),
               max_total_area_sample_mass=("signal_per_mass", "max"),
               mean_total_area_n_seeds=("signal_per_seed", "mean"),
               std_total_area_n_seeds=("signal_per_seed", "std"),
               min_total_area_n_seeds=("signal_per_seed", "min"),
               max_total_area_n_seeds=("signal_per_seed", "max"),
               mean_n_seeds=("n_seeds", "mean"),
               std_n_seeds=("n_seeds", "std"),
               n_replicates=(replicate_col, "nunique"),
           )
           .reset_index()
           .rename(columns={
               "mean_total_area_sample_mass": "mean_total_area/sample_mass",
               "std_total_area_sample_mass": "std_total_area/sample_mass",
               "min_total_area_sample_mass": "min_total_area/sample_mass",
               "max_total_area_sample_mass": "max_total_area/sample_mass",
               "mean_total_area_n_seeds": "mean_total_area/n_seeds",
               "std_total_area_n_seeds": "std_total_area/n_seeds",
               "min_total_area_n_seeds": "min_total_area/n_seeds",
               "max_total_area_n_seeds": "max_total_area/n_seeds",
           })
    )

    # merge compound count columns
    out = out.merge(compound_counts, on=[species_col, treatment_col], how="left")
    out["n_compounds_detected_>=_2_reps"] = out["n_compounds_detected_ge_2_reps"].fillna(0).astype(int)
    out["n_compounds_detected_3_reps"] = out["n_compounds_detected_3_reps"].fillna(0).astype(int)

    # ---- merge seed surface areas (species-level; same for all treatments) ----
    sa = (
        seed_area_df[seed_area_df[seed_area_species_col] != empty_label]
        .groupby(seed_area_species_col, dropna=False)
        .agg(
            mean_projected_surface_area=(projected_area_col, "mean"),
            std_projected_surface_area=(projected_area_col, "std"),
            mean_approx_surface_area=(approx_area_col, "mean"),
            std_approx_surface_area=(approx_area_col, "std"),
            n_area_measurements=(projected_area_col, "count"),
        )
        .reset_index()
        .rename(columns={seed_area_species_col: species_col})
    )

    out = out.merge(sa, on=species_col, how="left")

    # ---- merge seed traits directly (already one row per species) ----
    st = (
        seed_traits_df.loc[seed_traits_df[seed_traits_species_col] != empty_label, [
            seed_traits_species_col,
            seed_mass_col,
            oil_low_col,
            oil_high_col,
            oil_mean_col,
            germination_change_col,
        ]]
        .drop_duplicates(subset=[seed_traits_species_col])
        .rename(columns={
            seed_traits_species_col: species_col,
            seed_mass_col: "seed_mass_mg",
            oil_low_col: "oil_content_low",
            oil_high_col: "oil_content_high",
            oil_mean_col: "oil_content_mean",
            germination_change_col: "germination_change",
        })
    )

    out = out.merge(st, on=species_col, how="left")

    return out

all_summary = make_species_treatment_summary(gcms_df, seed_area_df, seed_traits_df)
oil_markers_summary = make_species_treatment_summary(gcms_df, seed_area_df, seed_traits_df, subset={"norm": oil_markers, "match": "exact"})
ageing_markers_summary = make_species_treatment_summary(gcms_df, seed_area_df, seed_traits_df, subset={"norm": ageing_markers, "match": "exact"})

In [9]:
# Wilcoxon signed-rank test for examining differences between treatments
def wilcoxon_species_treatment_test(
    summary_df,
    value_col,
    treatment_a,
    treatment_b,
    species_col="species",
    treatment_col="treatment",
    alternative="two-sided",
):
    
    # reshape so treatments become columns
    pivot = (
        summary_df
        .pivot(index=species_col, columns=treatment_col, values=value_col)
    )
    
    # keep only species present in both treatments
    paired = pivot[[treatment_a, treatment_b]].dropna()
    
    x = paired[treatment_a]
    y = paired[treatment_b]
    
    stat, p = wilcoxon(x, y, alternative=alternative)
    
    print(f"\nWilcoxon signed-rank test")
    print(f"{treatment_a} vs {treatment_b}")
    print(f"variable: {value_col}")
    print(f"n species = {len(paired)}")
    print(f"statistic = {stat:.3f}")
    print(f"p-value = {p:.5f}")
    
    return stat, p, paired

In [10]:
# vacuum vs normal atmosphere

print(wilcoxon_species_treatment_test(
    all_summary,
    value_col="mean_total_area/sample_mass",
    treatment_a="normal_atm_whole",
    treatment_b="vacuum_whole",
))

print(wilcoxon_species_treatment_test(
    all_summary,
    value_col="n_compounds_detected_ge_2_reps",
    treatment_a="normal_atm_whole",
    treatment_b="vacuum_whole",
))


Wilcoxon signed-rank test
normal_atm_whole vs vacuum_whole
variable: mean_total_area/sample_mass
n species = 7
statistic = 0.000
p-value = 0.01562
(np.float64(0.0), np.float64(0.015625), treatment                      normal_atm_whole  vacuum_whole
species                                                      
Daucus carota                      1.361667e+07  2.124839e+07
Ficus sycomorus                    1.911672e+05  1.316555e+07
Lactuca serriola                   3.019106e+05  2.258420e+06
Mesembryanthemum crystallinum      6.075728e+04  3.110935e+06
Papaver rhoeas                     3.606379e+05  1.291344e+07
Trifolium pratense                 4.914589e+04  1.989432e+06
Vaccinium oxycoccus                1.801724e+05  4.628852e+06)

Wilcoxon signed-rank test
normal_atm_whole vs vacuum_whole
variable: n_compounds_detected_ge_2_reps
n species = 7
statistic = 0.000
p-value = 0.01562
(np.float64(0.0), np.float64(0.015625), treatment                      normal_atm_whole  vacuum_whole


The amount of volatile emission per whole seed is significantly higher in a vacuum compared with normal atmosphere.
In addition, a significantly greater number of compounds are emitted in vacuum conditions compared with normal atmospheric conditions.

In [11]:
# vacuum vs normal atmosphere for marker compounds

print(wilcoxon_species_treatment_test(
    oil_markers_summary,
    value_col="mean_total_area/sample_mass",
    treatment_a="normal_atm_whole",
    treatment_b="vacuum_whole",
))

print(wilcoxon_species_treatment_test(
    ageing_markers_summary,
    value_col="mean_total_area/sample_mass",
    treatment_a="normal_atm_whole",
    treatment_b="vacuum_whole",
))


Wilcoxon signed-rank test
normal_atm_whole vs vacuum_whole
variable: mean_total_area/sample_mass
n species = 7
statistic = 0.000
p-value = 0.01562
(np.float64(0.0), np.float64(0.015625), treatment                      normal_atm_whole  vacuum_whole
species                                                      
Daucus carota                      2.920716e+06  3.147092e+06
Ficus sycomorus                    0.000000e+00  1.121393e+07
Lactuca serriola                   0.000000e+00  1.542241e+06
Mesembryanthemum crystallinum      0.000000e+00  2.072958e+06
Papaver rhoeas                     0.000000e+00  3.827458e+06
Trifolium pratense                 0.000000e+00  1.284783e+06
Vaccinium oxycoccus                0.000000e+00  2.221746e+06)

Wilcoxon signed-rank test
normal_atm_whole vs vacuum_whole
variable: mean_total_area/sample_mass
n species = 7
statistic = 0.000
p-value = 0.01562
(np.float64(0.0), np.float64(0.015625), treatment                      normal_atm_whole   vacuum_whole
sp

In [12]:
# vacuum whole vs vacuum cut

print(wilcoxon_species_treatment_test(
    all_summary,
    value_col="mean_total_area/n_seeds",
    treatment_a="vacuum_cut",
    treatment_b="vacuum_whole",
))

print(wilcoxon_species_treatment_test(
    all_summary,
    value_col="n_compounds_detected_ge_2_reps",
    treatment_a="vacuum_cut",
    treatment_b="vacuum_whole",
))


Wilcoxon signed-rank test
vacuum_cut vs vacuum_whole
variable: mean_total_area/n_seeds
n species = 7
statistic = 0.000
p-value = 0.01562
(np.float64(0.0), np.float64(0.015625), treatment                        vacuum_cut  vacuum_whole
species                                                  
Daucus carota                  5.444186e+07  1.820168e+07
Ficus sycomorus                1.281410e+07  7.387529e+06
Lactuca serriola               1.335961e+07  1.711480e+06
Mesembryanthemum crystallinum  2.164608e+06  5.578558e+05
Papaver rhoeas                 1.204271e+06  1.127486e+06
Trifolium pratense             9.415966e+06  2.968168e+06
Vaccinium oxycoccus            1.361257e+07  2.627988e+06)

Wilcoxon signed-rank test
vacuum_cut vs vacuum_whole
variable: n_compounds_detected_ge_2_reps
n species = 7
statistic = 0.000
p-value = 0.01562
(np.float64(0.0), np.float64(0.015625), treatment                      vacuum_cut  vacuum_whole
species                                                
Da

Volatile emission in a vacuum is significantly higher in cut seeds than whole seeds. In addition, a significantly greater number of compounds are emitted by cut seeds.

In [14]:
all_summary["log_mean_total_area/n_seeds"] = np.log10(all_summary["mean_total_area/n_seeds"] + 1)

# linear mixed model
model = smf.mixedlm(
    "Q('log_mean_total_area/n_seeds') ~ treatment",
    data=all_summary,
    groups=all_summary["species"]
).fit()

print(model.summary())

                    Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Q('log_mean_total_area/n_seeds')
No. Observations: 21      Method:             REML                            
No. Groups:       7       Scale:              0.1215                          
Min. group size:  3       Log-Likelihood:     -16.4430                        
Max. group size:  3       Converged:          Yes                             
Mean group size:  3.0                                                         
--------------------------------------------------------------------------------
                             Coef.   Std.Err.     z      P>|z|   [0.025   0.975]
--------------------------------------------------------------------------------
Intercept                    5.128      0.265   19.340   0.000    4.608    5.647
treatment[T.vacuum_cut]      1.800      0.186    9.659   0.000    1.435    2.165
treatment[T.vacuum_whole]    1.308      0.186    7.022   0.000 

Whole seeds under vacuum emit ~10^1.308 times (~20) times more volatiles than whole seeds under normal atmosphere.
Cutting seeds under vacuum increases emission to about ~63 times the level of whole seeds under normal atmosphere.
(Probably shouldn't quote quantitative results such as these in the paper due to compound-dependent interactions with the SPME fibre.)

In [28]:
# Wilcoxon test for comparing differences in the fraction of signal from marker compounds between treatments
def wilcoxon_species_treatment_test_fraction(
    df,
    compounds_of_interest,
    treatment_a,
    treatment_b,
    species_col="species",
    treatment_col="treatment",
    replicate_col="replicate",
    compound_col="compound_name",
    value_col="area/sample_mass",
    compound_norm_col="compound_norm",
    alternative="two-sided",
    exclude_species=("Empty",),
    return_intermediates=True,
):
    """
    Wilcoxon signed-rank test comparing treatments using the fraction of signal from selected compounds.

        per replicate fraction = selected_total / total_signal
        species-level value    = mean of replicate fractions

    Parameters
    ----------
    df : pd.DataFrame
        Raw long-form GC-MS dataframe.
    compounds_of_interest : list-like
        Marker compounds, ideally in the normalised form matching compound_norm_col.
    treatment_a, treatment_b : str
        Treatments to compare.
    species_col, treatment_col, replicate_col, compound_col : str
        Column names.
    value_col : str
        Signal column, e.g. "area/sample_mass" or "area/n_seeds".
    compound_norm_col : str
        Normalised compound-name column used to identify marker compounds.
    alternative : {"two-sided", "greater", "less"}
        Passed to scipy.stats.wilcoxon.
    exclude_species : tuple
        Species to exclude, e.g. ("Empty",).
    return_intermediates : bool
        If True, also return replicate-level and species-level summary dfs.

    Returns
    -------
    stat : float
    p : float
    paired : pd.DataFrame
        One row per species used in the Wilcoxon test.
    rep_base : pd.DataFrame, optional
        Replicate-level totals and fractions.
    species_fraction : pd.DataFrame, optional
        Species-treatment mean replicate fractions.
    """

    d = df.copy()

    # keep only relevant treatments/species
    d = d[d[treatment_col].isin([treatment_a, treatment_b])].copy()
    if exclude_species is not None:
        d = d[~d[species_col].isin(exclude_species)].copy()

    # ----------------------------
    # 1) Replicate-level total signal
    # ----------------------------
    rep_total = (
        d.groupby([species_col, treatment_col, replicate_col], dropna=False)[value_col]
         .sum()
         .rename("total_signal")
         .reset_index()
    )

    # ----------------------------
    # 2) Identify selected compounds
    # ----------------------------
    if compound_norm_col in d.columns:
        selected_mask = d[compound_norm_col].isin(compounds_of_interest)
        selected_compounds = (
            d.loc[selected_mask, compound_col]
             .dropna()
             .unique()
             .tolist()
        )
    else:
        selected_compounds = list(compounds_of_interest)

    # ----------------------------
    # 3) Replicate-level selected total
    # ----------------------------
    rep_selected = (
        d[d[compound_col].isin(selected_compounds)]
        .groupby([species_col, treatment_col, replicate_col], dropna=False)[value_col]
        .sum()
        .rename("selected_total")
        .reset_index()
    )

    # merge so missing selected compounds become 0
    rep_base = rep_total.merge(
        rep_selected,
        on=[species_col, treatment_col, replicate_col],
        how="left"
    )
    rep_base["selected_total"] = rep_base["selected_total"].fillna(0.0)

    # replicate-level fraction
    rep_base["fraction"] = np.where(
        rep_base["total_signal"] > 0,
        rep_base["selected_total"] / rep_base["total_signal"],
        np.nan
    )

    # ----------------------------
    # 4) Species-level mean replicate fraction
    # ----------------------------
    species_fraction = (
        rep_base.groupby([species_col, treatment_col], as_index=False)["fraction"]
        .mean()
        .rename(columns={"fraction": "mean_fraction"})
    )

    # reshape for paired Wilcoxon
    pivot = species_fraction.pivot(
        index=species_col,
        columns=treatment_col,
        values="mean_fraction"
    )

    paired = pivot[[treatment_a, treatment_b]].dropna().copy()

    x = paired[treatment_a]
    y = paired[treatment_b]

    stat, p = wilcoxon(x, y, alternative=alternative)

    print("\nWilcoxon signed-rank test on marker fraction")
    print(f"{treatment_a} vs {treatment_b}")
    print(f"fraction based on replicate-level: selected_total / total_signal")
    print(f"species-level value: mean replicate fraction using {value_col}")
    print(f"n species = {len(paired)}")
    print(f"statistic = {stat:.3f}")
    print(f"p-value = {p:.5f}")

    if return_intermediates:
        return stat, p, paired, rep_base, species_fraction
    else:
        return stat, p, paired

In [31]:
print("Oil markers:")
stat, p, paired, rep_base, species_fraction = wilcoxon_species_treatment_test_fraction(
    gcms_df,
    compounds_of_interest=oil_markers,
    treatment_a="vacuum_whole",
    treatment_b="normal_atm_whole",
    value_col="area/sample_mass", 
    alternative="two-sided",
)
print(species_fraction)

Oil markers:

Wilcoxon signed-rank test on marker fraction
vacuum_whole vs normal_atm_whole
fraction based on replicate-level: selected_total / total_signal
species-level value: mean replicate fraction using area/sample_mass
n species = 7
statistic = 1.000
p-value = 0.03125
                          species         treatment  mean_fraction
0                   Daucus carota  normal_atm_whole       0.212469
1                   Daucus carota      vacuum_whole       0.139873
2                 Ficus sycomorus  normal_atm_whole       0.000000
3                 Ficus sycomorus      vacuum_whole       0.852493
4                Lactuca serriola  normal_atm_whole       0.000000
5                Lactuca serriola      vacuum_whole       0.690901
6   Mesembryanthemum crystallinum  normal_atm_whole       0.000000
7   Mesembryanthemum crystallinum      vacuum_whole       0.690655
8                  Papaver rhoeas  normal_atm_whole       0.000000
9                  Papaver rhoeas      vacuum_whole    

In [33]:
print("Oil markers:")
stat, p, paired, rep_base, species_fraction = wilcoxon_species_treatment_test_fraction(
    gcms_df,
    compounds_of_interest=oil_markers,
    treatment_a="vacuum_whole",
    treatment_b="vacuum_cut",
    value_col="area/sample_mass", 
    alternative="two-sided",
)
print(species_fraction)

Oil markers:

Wilcoxon signed-rank test on marker fraction
vacuum_whole vs vacuum_cut
fraction based on replicate-level: selected_total / total_signal
species-level value: mean replicate fraction using area/sample_mass
n species = 7
statistic = 5.000
p-value = 0.15625
                          species     treatment  mean_fraction
0                   Daucus carota    vacuum_cut       0.325620
1                   Daucus carota  vacuum_whole       0.139873
2                 Ficus sycomorus    vacuum_cut       0.860438
3                 Ficus sycomorus  vacuum_whole       0.852493
4                Lactuca serriola    vacuum_cut       0.745665
5                Lactuca serriola  vacuum_whole       0.690901
6   Mesembryanthemum crystallinum    vacuum_cut       0.674035
7   Mesembryanthemum crystallinum  vacuum_whole       0.690655
8                  Papaver rhoeas    vacuum_cut       0.389520
9                  Papaver rhoeas  vacuum_whole       0.297613
10             Trifolium pratense    v

In [34]:
print("Ageing markers:")
stat, p, paired, rep_base, species_fraction = wilcoxon_species_treatment_test_fraction(
    gcms_df,
    compounds_of_interest=ageing_markers,
    treatment_a="vacuum_whole",
    treatment_b="normal_atm_whole",
    value_col="area/sample_mass", 
    alternative="two-sided",
)
print(species_fraction)

Ageing markers:

Wilcoxon signed-rank test on marker fraction
vacuum_whole vs normal_atm_whole
fraction based on replicate-level: selected_total / total_signal
species-level value: mean replicate fraction using area/sample_mass
n species = 7
statistic = 1.000
p-value = 0.03125
                          species         treatment  mean_fraction
0                   Daucus carota  normal_atm_whole       0.000000
1                   Daucus carota      vacuum_whole       0.013165
2                 Ficus sycomorus  normal_atm_whole       0.000000
3                 Ficus sycomorus      vacuum_whole       0.036987
4                Lactuca serriola  normal_atm_whole       0.000000
5                Lactuca serriola      vacuum_whole       0.113921
6   Mesembryanthemum crystallinum  normal_atm_whole       0.000000
7   Mesembryanthemum crystallinum      vacuum_whole       0.083028
8                  Papaver rhoeas  normal_atm_whole       0.015166
9                  Papaver rhoeas      vacuum_whole 

In [35]:
print("Ageing markers:")
stat, p, paired, rep_base, species_fraction = wilcoxon_species_treatment_test_fraction(
    gcms_df,
    compounds_of_interest=ageing_markers,
    treatment_a="vacuum_whole",
    treatment_b="vacuum_cut",
    value_col="area/sample_mass", 
    alternative="two-sided",
)
print(species_fraction)

Ageing markers:

Wilcoxon signed-rank test on marker fraction
vacuum_whole vs vacuum_cut
fraction based on replicate-level: selected_total / total_signal
species-level value: mean replicate fraction using area/sample_mass
n species = 7
statistic = 6.000
p-value = 0.21875
                          species     treatment  mean_fraction
0                   Daucus carota    vacuum_cut       0.008825
1                   Daucus carota  vacuum_whole       0.013165
2                 Ficus sycomorus    vacuum_cut       0.018746
3                 Ficus sycomorus  vacuum_whole       0.036987
4                Lactuca serriola    vacuum_cut       0.033376
5                Lactuca serriola  vacuum_whole       0.113921
6   Mesembryanthemum crystallinum    vacuum_cut       0.087536
7   Mesembryanthemum crystallinum  vacuum_whole       0.083028
8                  Papaver rhoeas    vacuum_cut       0.034716
9                  Papaver rhoeas  vacuum_whole       0.005918
10             Trifolium pratense  

In [16]:
# get replicate-level totals
rep_df = (
    gcms_df
    .query("species != 'Empty'")
    .groupby(["species", "treatment", "replicate"], as_index=False)
    .agg(
        total_area=("area", "sum"),
        n_seeds=("n_seeds", "first"),
        sample_mass=("sample_mass_mg", "first")
    )
)

rep_df["total_area/sample_mass"] = rep_df["total_area"] / rep_df["sample_mass"]
rep_df["total_area/n_seeds"] = rep_df["total_area"] / rep_df["n_seeds"]

In [17]:
# Perform Kruskal-Wallis test to assess if there are species-level differences in each treatment

print("Vacuum (whole):")
subset = rep_df[rep_df["treatment"] == "vacuum_whole"]
groups = [
    g["total_area/sample_mass"].values
    for _, g in subset.groupby("species")
]
stat, p = kruskal(*groups)

print(f"KW stat: {stat}, p-value: {p}")

print("\nVacuum (cut):")
subset = rep_df[rep_df["treatment"] == "vacuum_cut"]
groups = [
    g["total_area/sample_mass"].values
    for _, g in subset.groupby("species")
]
stat, p = kruskal(*groups)

print(f"KW stat: {stat}, p-value: {p}")

Vacuum (whole):
KW stat: 17.385281385281388, p-value: 0.007966850408673854

Vacuum (cut):
KW stat: 17.004329004329023, p-value: 0.009267347767073691


In [18]:
# Perform follow-up Dunn's test for pairwise species comparisons within each treatment

subset = rep_df[rep_df["treatment"] == "vacuum_whole"]
dunn = sp.posthoc_dunn(
    subset,
    val_col="total_area/sample_mass",
    group_col="species",
    p_adjust="fdr_bh"
)
pairs = (
    dunn.stack()
    .reset_index()
    .rename(columns={0:"p"})
)
pairs = pairs[pairs["level_0"] < pairs["level_1"]]  # remove duplicates
pairs = pairs[pairs["p"] < 0.05]
print("Significant pairwise differences (Dunn's test, vacuum whole):")
print(pairs)


subset = rep_df[rep_df["treatment"] == "vacuum_cut"]
dunn = sp.posthoc_dunn(
    subset,
    val_col="total_area/sample_mass",
    group_col="species",
    p_adjust="fdr_bh"
)
pairs = (
    dunn.stack()
    .reset_index()
    .rename(columns={0:"p"})
)
pairs = pairs[pairs["level_0"] < pairs["level_1"]]  # remove duplicates
pairs = pairs[pairs["p"] < 0.05]
print("\nSignificant pairwise differences (Dunn's test, vacuum cut):")
print(pairs)

Significant pairwise differences (Dunn's test, vacuum whole):
         level_0             level_1         p
2  Daucus carota    Lactuca serriola  0.039813
5  Daucus carota  Trifolium pratense  0.039813

Significant pairwise differences (Dunn's test, vacuum cut):
         level_0             level_1      p
5  Daucus carota  Trifolium pratense  0.008


In [19]:
# Perform Kruskal-Wallis test on oil marker subset to assess species-level differences within each treatment

oil_df = gcms_df[
    gcms_df["compound_norm"].isin(oil_markers)
].copy()

oil_rep_df = (
    oil_df
    .query("species != 'Empty'")
    .groupby(["species", "treatment", "replicate"], as_index=False)
    .agg(
        total_area=("area", "sum"),
        n_seeds=("n_seeds", "first"),
        sample_mass=("sample_mass_mg", "first")
    )
)

oil_rep_df["total_area/sample_mass"] = oil_rep_df["total_area"] / oil_rep_df["sample_mass"]
oil_rep_df["total_area/n_seeds"] = oil_rep_df["total_area"] / oil_rep_df["n_seeds"]

print("Vacuum (whole):")
subset = oil_rep_df[oil_rep_df["treatment"] == "vacuum_whole"]
groups = [
    g["total_area/sample_mass"].values
    for _, g in subset.groupby("species")
]
stat, p = kruskal(*groups)

print(f"KW stat: {stat}, p-value: {p}")

print("\nVacuum (cut):")
subset = oil_rep_df[oil_rep_df["treatment"] == "vacuum_cut"]
groups = [
    g["total_area/sample_mass"].values
    for _, g in subset.groupby("species")
]
stat, p = kruskal(*groups)

print(f"KW stat: {stat}, p-value: {p}")

Vacuum (whole):
KW stat: 12.484848484848499, p-value: 0.051986349438988944

Vacuum (cut):
KW stat: 17.281385281385298, p-value: 0.008302886203702958


In [21]:
# Follow-up Dunn's test for pairwise species comparisons within oil marker subset

subset = oil_rep_df[oil_rep_df["treatment"] == "vacuum_whole"]
dunn = sp.posthoc_dunn(
    subset,
    val_col="total_area/sample_mass",
    group_col="species",
    p_adjust="fdr_bh"
)
pairs = (
    dunn.stack()
    .reset_index()
    .rename(columns={0:"p"})
)
pairs = pairs[pairs["level_0"] < pairs["level_1"]]  # remove duplicates
pairs = pairs[pairs["p"] < 0.05]
print("Significant pairwise differences (Dunn's test, vacuum whole):")
print(pairs)


subset = oil_rep_df[oil_rep_df["treatment"] == "vacuum_cut"]
dunn = sp.posthoc_dunn(
    subset,
    val_col="total_area/sample_mass",
    group_col="species",
    p_adjust="fdr_bh"
)
pairs = (
    dunn.stack()
    .reset_index()
    .rename(columns={0:"p"})
)
pairs = pairs[pairs["level_0"] < pairs["level_1"]]  # remove duplicates
pairs = pairs[pairs["p"] < 0.05]
print("\nSignificant pairwise differences (Dunn's test, vacuum cut):")
print(pairs)

Significant pairwise differences (Dunn's test, vacuum whole):
Empty DataFrame
Columns: [level_0, level_1, p]
Index: []

Significant pairwise differences (Dunn's test, vacuum cut):
               level_0              level_1         p
5        Daucus carota   Trifolium pratense  0.048889
12     Ficus sycomorus   Trifolium pratense  0.048889
41  Trifolium pratense  Vaccinium oxycoccus  0.048889


In [20]:
# Perform Kruskal-Wallis test on ageing marker subset to assess species-level differences within each treatment

ageing_df = gcms_df[
    gcms_df["compound_norm"].isin(ageing_markers)
].copy()

ageing_rep_df = (
    ageing_df
    .query("species != 'Empty'")
    .groupby(["species", "treatment", "replicate"], as_index=False)
    .agg(
        total_area=("area", "sum"),
        n_seeds=("n_seeds", "first"),
        sample_mass=("sample_mass_mg", "first")
    )
)

ageing_rep_df["total_area/sample_mass"] = ageing_rep_df["total_area"] / ageing_rep_df["sample_mass"]
ageing_rep_df["total_area/n_seeds"] = ageing_rep_df["total_area"] / ageing_rep_df["n_seeds"]

print("Vacuum (whole):")
subset = ageing_rep_df[ageing_rep_df["treatment"] == "vacuum_whole"]
groups = [
    g["total_area/sample_mass"].values
    for _, g in subset.groupby("species")
]
stat, p = kruskal(*groups)

print(f"KW stat: {stat}, p-value: {p}")

print("\nVacuum (cut):")
subset = ageing_rep_df[ageing_rep_df["treatment"] == "vacuum_cut"]
groups = [
    g["total_area/sample_mass"].values
    for _, g in subset.groupby("species")
]
stat, p = kruskal(*groups)

print(f"KW stat: {stat}, p-value: {p}")

Vacuum (whole):
KW stat: 8.22510822510823, p-value: 0.22207089269223473

Vacuum (cut):
KW stat: 2.7532467532467706, p-value: 0.8391183202433239


In [22]:
# get compound masses for each treatment

df = gcms_df.query("species != 'Empty'").copy()

cols = ["compound_norm", "species", "treatment", "mass_db"]
df = df[cols]

df_unique = (
    df
    .drop_duplicates(subset=["compound_norm", "species", "treatment"])
)

vw_df = df_unique[df_unique["treatment"] == "vacuum_whole"]
vc_df   = df_unique[df_unique["treatment"] == "vacuum_cut"]
na_df = df_unique[df_unique["treatment"] == "normal_atm_whole"]

vw_masses = vw_df['mass_db'].dropna()
vc_masses = vc_df['mass_db'].dropna()
na_masses = na_df['mass_db'].dropna()

In [ ]:
# Perform Mann-Whitney U test to compare compound mass distributions between treatments

print("Vacuum whole vs Normal atmosphere whole:")
U, p = mannwhitneyu(
    vw_masses,
    na_masses,
    alternative="greater"
) 

print(f"Test statistic: {U}, p-value (null hypothesis that compound mass distributions are the same): {p}")
P = U / (len(vw_masses)*len(na_masses))
print(f"Probability of randomly selecting a compound of larger mass from vacuum whole than normal atmosphere whole: {P}")

print("\nVacuum whole vs Vacuum cut:")
U, p = mannwhitneyu(
    vw_masses,
    vc_masses,
    alternative="greater"
)

print(f"Test statistic: {U}, p-value (null hypothesis that compound mass distributions are the same): {p}")
P = U / (len(vw_masses)*len(vc_masses))
print(f"Probability of randomly selecting a compound of larger mass from vacuum whole than vacuum cut: {P}")

Vacuum whole vs Normal atmosphere whole:
Test statistic: 7380.5, p-value (null hypothesis that compound mass distributions are the same): 3.9447283013341184e-09
Probability of randomly selecting a compound of larger mass from vacuum whole than normal atmosphere whole: 0.7648186528497409

Vacuum whole vs Vacuum cut:
Test statistic: 35674.5, p-value (null hypothesis that compound mass distributions are the same): 6.62176238943632e-06
Probability of randomly selecting a compound of larger mass from vacuum whole than vacuum cut: 0.616139896373057


In [26]:
# For each species, compare compound mass distributions between treatments using Mann-Whitney U test

print("Vacuum whole vs Normal atmosphere whole, by species:")
df_unique = (
    gcms_df
    .query("species != 'Empty'")
    .drop_duplicates(subset=["compound_norm","species","treatment"])
    [["compound_norm","species","treatment","mass_db"]]
)

results = []

for sp, g in df_unique.groupby("species"):

    whole = g.loc[g["treatment"]=="vacuum_whole","mass_db"].dropna()
    normal = g.loc[g["treatment"]=="normal_atm_whole","mass_db"].dropna()

    if len(whole) > 0 and len(normal) > 0:

        U, p = mannwhitneyu(whole, normal, alternative="greater")

        results.append({
            "species": sp,
            "n_whole": len(whole),
            "n_normal": len(normal),
            "P (m_vac > m_normal)": U / (len(whole) * len(normal)),
            "p": p
        })

results_df = pd.DataFrame(results)

results_df["p_adj"] = multipletests(
    results_df["p"],
    method="fdr_bh"
    )[1]

print(results_df)

Vacuum whole vs Normal atmosphere whole, by species:
                         species  n_whole  n_normal  P (m_vac > m_normal)  \
0                  Daucus carota       43        31              0.682296   
1                Ficus sycomorus       33         6              0.840909   
2               Lactuca serriola       22         2              0.965909   
3  Mesembryanthemum crystallinum       23         1              1.000000   
4                 Papaver rhoeas       33         6              0.785354   
5             Trifolium pratense       14         1              1.000000   
6            Vaccinium oxycoccus       25         3              0.833333   

          p     p_adj  
0  0.003779  0.015829  
1  0.004522  0.015829  
2  0.018112  0.031696  
3  0.041667  0.048611  
4  0.014457  0.031696  
5  0.066667  0.066667  
6  0.034090  0.047726  


In [27]:
print("Vacuum whole vs Vacuum cut, by species:")

results = []

for sp, g in df_unique.groupby("species"):

    whole = g.loc[g["treatment"]=="vacuum_whole","mass_db"].dropna()
    cut   = g.loc[g["treatment"]=="vacuum_cut","mass_db"].dropna()

    if len(whole) > 0 and len(cut) > 0:

        U, p = mannwhitneyu(whole, cut, alternative="greater")

        results.append({
            "species": sp,
            "n_whole": len(whole),
            "n_cut": len(cut),
            "P (m_whole > m_cut)": U / (len(whole) * len(cut)),
            "p": p
        })

results_df = pd.DataFrame(results)

results_df["p_adj"] = multipletests(
    results_df["p"],
    method="fdr_bh"
    )[1]

print(results_df)

Vacuum whole vs Vacuum cut, by species:
                         species  n_whole  n_cut  P (m_whole > m_cut)  \
0                  Daucus carota       43     56             0.580357   
1                Ficus sycomorus       33     35             0.646753   
2               Lactuca serriola       22     49             0.715677   
3  Mesembryanthemum crystallinum       23     39             0.571906   
4                 Papaver rhoeas       33     48             0.568182   
5             Trifolium pratense       14     29             0.597291   
6            Vaccinium oxycoccus       25     44             0.596364   

          p     p_adj  
0  0.086147  0.163769  
1  0.018955  0.066344  
2  0.001949  0.013645  
3  0.175381  0.175381  
4  0.150498  0.175381  
5  0.155919  0.175381  
6  0.093582  0.163769  


In [ ]:
stat, p, paired, rep_base, species_fraction = wilcoxon_species_treatment_test_fraction(
    gcms_df,
    compounds_of_interest=ageing_markers,
    treatment_a="vacuum_whole",
    treatment_b="vacuum_cut",
    value_col="area/sample_mass",   # or "area/sample_mass"
    alternative="two-sided",
)

print(species_fraction)


Wilcoxon signed-rank test on marker fraction
vacuum_whole vs vacuum_cut
fraction based on replicate-level: selected_total / total_signal
species-level value: mean replicate fraction using area/sample_mass
n species = 7
statistic = 6.000
p-value = 0.21875
                          species     treatment  mean_fraction
0                   Daucus carota    vacuum_cut       0.008825
1                   Daucus carota  vacuum_whole       0.013165
2                 Ficus sycomorus    vacuum_cut       0.018746
3                 Ficus sycomorus  vacuum_whole       0.036987
4                Lactuca serriola    vacuum_cut       0.033376
5                Lactuca serriola  vacuum_whole       0.113921
6   Mesembryanthemum crystallinum    vacuum_cut       0.087536
7   Mesembryanthemum crystallinum  vacuum_whole       0.083028
8                  Papaver rhoeas    vacuum_cut       0.034716
9                  Papaver rhoeas  vacuum_whole       0.005918
10             Trifolium pratense    vacuum_cut    